# UNSW-NB15: перевірка складу атак у temporal split
UNSW-NB15 використовує Stime (Unix-час), тож проблеми з AM/PM тут бути не повинно —
ця перевірка саме про те, чи немає, як у CIC-IDS2017, класів атак, відсутніх у train
але присутніх у val/test. Нічого не змінює у ваших файлах — лише читає і друкує.

In [1]:
# 0. Автопошук файлу на сервері
import subprocess

result = subprocess.run(
    ["find", "/srv/ids_research", "/home/ivan_r_opirskyy", "-iname", "*unsw_nb15*", "-type", "f"],
    capture_output=True, text=True, timeout=60
)
print(result.stdout)
print(result.stderr)

/srv/ids_research/data/enriched/unsw_nb15_with_cve_context_dedup.parquet
/srv/ids_research/data/enriched/unsw_nb15_with_cve_context.parquet




In [2]:
# 1. Налаштування — підтверджений шлях (дедуплікований, CTI-збагачений файл, зафіксований у проєкті)
import pandas as pd, numpy as np

PATH = "/srv/ids_research/data/enriched/unsw_nb15_with_cve_context_dedup.parquet"
TS_COL = "Stime"             # Unix-час (секунди), згідно з проєктною фіксацією
LABEL_COL_CANDIDATES = ["attack_cat", "Label", "label"]

df = pd.read_parquet(PATH) if PATH.endswith(".parquet") else pd.read_csv(PATH, low_memory=False)
df.columns = [c.strip() for c in df.columns]
LABEL_COL = next(c for c in LABEL_COL_CANDIDATES if c in df.columns)
print("rows:", len(df), "| label column:", LABEL_COL)
print(df[LABEL_COL].value_counts().to_string())

rows: 2059418 | label column: attack_cat
attack_cat
Normal            1959775
Exploits            27599
Generic             25378
Fuzzers             21795
Reconnaissance      13357
DoS                  5665
Analysis             2184
Backdoor             1684
Shellcode            1511
Backdoors             299
Worms                 171


In [3]:
# 2. Парсинг Stime і базова перевірка діапазону
raw = df[TS_COL]
print("dtype:", raw.dtype)
print(raw.sample(10, random_state=0).to_string())

ts = pd.to_datetime(raw, unit="s", errors="coerce")
print("\nне розпарсилось:", ts.isna().sum())
df["_ts"] = ts
print("діапазон:", ts.min(), "—", ts.max())

dtype: int64
786352     1421962493
1245475    1424229885
853416     1421965198
1744532    1424249682
1048328    1424221992
1088199    1424223521
1039714    1424221599
1189676    1424227706
1507696    1424240378
1040592    1424221618

не розпарсилось: 0
діапазон: 2015-01-22 11:49:37 — 2015-02-18 12:21:08


In [4]:
# 3. Час кожного класу атак
g = df.groupby(LABEL_COL)["_ts"].agg(["min", "max", "count"])
print(g.to_string())

                               min                 max    count
attack_cat                                                     
Analysis       2015-01-22 12:43:22 2015-02-18 12:16:22     2184
Backdoor       2015-02-18 00:24:20 2015-02-18 12:19:48     1684
Backdoors      2015-01-22 12:01:34 2015-01-22 13:45:17      299
DoS            2015-01-22 11:50:19 2015-02-18 12:21:06     5665
Exploits       2015-01-22 11:50:15 2015-02-18 12:21:08    27599
Fuzzers        2015-01-22 11:53:16 2015-02-18 12:21:07    21795
Generic        2015-01-22 11:50:20 2015-02-18 12:21:08    25378
Normal         2015-01-22 11:49:37 2015-02-18 12:21:08  1959775
Reconnaissance 2015-01-22 11:50:14 2015-02-18 12:21:07    13357
Shellcode      2015-01-22 11:51:05 2015-02-18 12:21:01     1511
Worms          2015-01-22 11:54:55 2015-02-18 12:12:54      171


In [5]:
# 4. Відтворення спліту 70/85 і склад атак по партиціях
def split_by_percentile(t):
    v = np.sort(t.dropna().values)
    b1 = v[int(round(0.70 * (len(v) - 1)))]
    b2 = v[int(round(0.85 * (len(v) - 1)))]
    part = np.where(t < b1, "train", np.where(t < b2, "val", "test"))
    return pd.Series(part, index=t.index), pd.Timestamp(b1), pd.Timestamp(b2)

df["_split"], b1, b2 = split_by_percentile(df["_ts"])
print("межі:", b1, "|", b2)
ct = pd.crosstab(df[LABEL_COL], df["_split"])
for col in ["train", "val", "test"]:
    if col not in ct.columns:
        ct[col] = 0
ct = ct[["train", "val", "test"]]
print(ct.to_string())

print("\nКласи атак, яких НЕМАЄ в train, але є у val/test:")
benign = [c for c in ct.index if str(c).lower() in ("normal", "benign", "0")]
problem = ct[(ct["train"] == 0) & ((ct["val"] + ct["test"]) > 0)].drop(index=benign, errors="ignore")
print(problem.to_string() if len(problem) else "— немає, усі класи присутні в train.")

межі: 2015-02-18 05:36:51 | 2015-02-18 08:58:36
_split            train     val    test
attack_cat                             
Analysis           1158     405     621
Backdoor            658     402     624
Backdoors           299       0       0
DoS                2952    1237    1476
Exploits          13901    6802    6896
Fuzzers           11268    5555    4972
Generic           11573    6903    6902
Normal          1392222  283972  283581
Reconnaissance     6686    3223    3448
Shellcode           763     375     373
Worms                88      41      42

Класи атак, яких НЕМАЄ в train, але є у val/test:
— немає, усі класи присутні в train.


In [6]:
# 5. Частка кожного класу в val/test відносно train (чи є суттєвий дисбаланс композиції,
# навіть якщо клас присутній в усіх трьох партиціях)
share = ct.div(ct.sum(axis=0), axis=1) * 100
print(share.round(2).to_string())
print("\n(Порівняйте частки по рядках — якщо якийсь клас у test/val на порядок частіший,")
print("ніж у train, це теж варто врахувати в інтерпретації ΔAP, навіть без формального 'відсутній у train'.)")

_split          train    val   test
attack_cat                         
Analysis         0.08   0.13   0.20
Backdoor         0.05   0.13   0.20
Backdoors        0.02   0.00   0.00
DoS              0.20   0.40   0.48
Exploits         0.96   2.20   2.23
Fuzzers          0.78   1.80   1.61
Generic          0.80   2.23   2.23
Normal          96.58  91.93  91.79
Reconnaissance   0.46   1.04   1.12
Shellcode        0.05   0.12   0.12
Worms            0.01   0.01   0.01

(Порівняйте частки по рядках — якщо якийсь клас у test/val на порядок частіший,
ніж у train, це теж варто врахувати в інтерпретації ΔAP, навіть без формального 'відсутній у train'.)
